# A Dataset Is Not a Population — Reproducible Experiments

Article #34 companion. All nine studies are **synthetic**. This notebook executes three independent replications per study in a temporary directory. It never changes canonical publication CSVs or PNGs. The complete 300-repetition study uses `analysis.py`.

In [ ]:
from pathlib import Path
import os
import sys
import tempfile

candidates = []
if os.environ.get('POPULATION_ARTICLE_DIR'):
    candidates.append(Path(os.environ['POPULATION_ARTICLE_DIR']))
candidates.extend([Path.cwd(), Path.cwd() / 'a-dataset-is-not-a-population'])
article_dir = next((p.resolve() for p in candidates if (p / 'analysis.py').is_file() and (p / 'figures.py').is_file()), None)
if article_dir is None:
    raise FileNotFoundError('Set POPULATION_ARTICLE_DIR to the companion directory')
sys.path.insert(0, str(article_dir))
from analysis import SEED, STUDIES, METHODS, simulate, write_evidence
from figures import render_all
print(f'Synthetic studies: {len(STUDIES)}, canonical seed: {SEED}')

## Reproduce the study contracts

The temporary run checks method counts, replication identities, CSV evidence and plot generation. It is a **smoke test**, not a substitute for the full archived Monte Carlo estimates.

In [ ]:
with tempfile.TemporaryDirectory(prefix='population-notebook-') as directory:
    output = Path(directory)
    for study in STUDIES:
        rows = simulate(study, repetitions=3, seed=SEED)
        assert len(rows) == 3 * len(METHODS[study])
        write_evidence(rows, output / 'data', '_quick')
    figures = render_all(output / 'data', output / 'figures', '_quick')
    assert len(figures) == 9 and all(p.stat().st_size > 5000 for p in figures)
    assert len(list((output / 'data').glob('*_quick.csv'))) == 18
    print('9 studies, 18 isolated CSVs, 9 isolated PNGs: PASS')

## Interpretation and limits

- The mean errors are conditional on specified synthetic mechanisms.
- Inverse response probabilities using latent outcomes are an oracle comparison, not an actionable estimator from respondents alone.
- Marginal calibration does not identify missing joint distributions.
- Covariate weighting requires support and conditional invariance.
- Composition updates cannot correct changing conditional outcomes.
- Monte Carlo confidence intervals around simulated bias are not population confidence intervals.